In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.course_planner import solve_csp

result = solve_csp(focus_area="Artificial Intelligence")

Semester:1
MA1521
GEN
ID1
ID2
CD
Total creds:20
Semester:2
MA1522
ST2334
GEC
GEA
GESS
Total creds:20
Semester:3
CS1101S
CS1231S
CS3245
CS5344
ES2660
IS1108
Total creds:21
Semester:4
CS2030S
CS2040S
CS2111
CS3234
CS3236
CS5224
Total creds:24
Semester:5
CS2104
CS2105
CS2107
CS2109S
CS3230
CS3241
Total creds:24
Semester:6
CS2100
CS2101
CS2103T
CS3263
CS3264
CS4248
Total creds:24
Semester:7
CP1001
CP3107
CP4281
CP4282
CP4283
CP4285
Total creds:24
Semester:8
CP1002
CP1008
CP4106
CP4284
CS2102
CS2106
Total creds:24


The following code is to fetch data from the nusmods api. 

Warning: It fetches ALL the courses data and takes around 20-30 mins to finish executing. Do not run unless sure you want it all. The `nusmods_modules.json` file already has the necessary courses for the first version of the project

In [ ]:
import json
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

BASE_URL = "https://api.nusmods.com/v2"
ACAD_YEAR = "2026-2027"
OUTPUT_FILE = Path("nusmods_modules.json")
TIMEOUT = 30


def get_json(url):
    response = requests.get(url, timeout=TIMEOUT)
    response.raise_for_status()
    return response.json()


def fetch_module(code):
    url = f"{BASE_URL}/{ACAD_YEAR}/modules/{code}.json"

    try:
        details = get_json(url)

        semester_data = details.get("semesterData", [])

        # Keep one entry per module, with its semesters in a list.
        return {
            "module_code": details.get("moduleCode", code),
            "module_credit": details.get("moduleCredit"),
            "semesters": [
                {
                    "semester_number": sem.get("semester"),
                    "exam_date_time": sem.get("examDate"),
                    "exam_duration": sem.get("examDuration"),
                }
                for sem in semester_data
            ],
            "preclusions": details.get("preclusion"),
            "prereq_tree": details.get("prereqTree"),
        }

    except requests.RequestException as error:
        print(f"Failed to fetch {code}: {error}")
        return None


print(json.dumps(fetch_module("CS3242"), indent=4))
